<a href="https://colab.research.google.com/github/dhillonarsh-ai/capstone-project-NoShowGuard-/blob/main/notebooks/Part4_AI_Follow_up_Note_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 4.1 Dataset Loading and Gemini API Setup

We begin by loading the patient follow-up notes dataset and preparing the environment for Gemini API integration.


In [29]:

import pandas as pd
import numpy as np

# Load follow-up notes
notes_url = "https://raw.githubusercontent.com/dhillonarsh-ai/capstone-project-NoShowGuard-/main/data/raw/followup_notes.csv"

notes_df = pd.read_csv(notes_url)

# Display dataset information
print("Dataset Shape:", notes_df.shape)
print("\nColumns:")
print(notes_df.columns.tolist())

print("\nFirst 5 Records:")
display(notes_df.head())

print("\nMissing Values:")
print(notes_df.isnull().sum())


Dataset Shape: (30, 3)

Columns:
['note_id', 'appointment_id', 'note_text']

First 5 Records:


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."



Missing Values:
note_id           0
appointment_id    0
note_text         1
dtype: int64


In [30]:

# Install required libraries
%pip install -q google-genai pydantic

## 4.2 Gemini API Configuration

Google Gemini is a Generative AI model used to understand and analyze natural language text.

In this project, Gemini is integrated through the Google GenAI Python SDK to analyze patient follow-up notes and extract meaningful insights about appointment no-shows.


In [31]:
from google import genai
from google.colab import userdata

# Read API key from Colab Secrets using the name/label of the secret
api_key = userdata.get("GEMINI_API_KEY")

# Initialize Gemini
client = genai.Client(api_key=api_key)

print("Gemini API configured successfully.")



Gemini API configured successfully.


In [34]:

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Reply with exactly: NoShowGuard API connected"
)

print(response.text)


NoShowGuard API connected


## 4.3 Structured Prompt Engineering and AI Note Analysis

In this section, we develop a structured prompt to analyze patient follow-up notes using Google Gemini.

The model identifies the potential reason for a missed appointment, assigns a confidence score, generates a concise summary, and recommends a suitable follow-up action.

Pydantic is used to validate the AI-generated response and maintain a consistent output structure.

In [35]:

from pydantic import BaseModel, Field
from typing import Literal
import json

class FollowupInsight(BaseModel):

    root_cause: Literal[
        "No Reminder",
        "Distance",
        "Cost",
        "Health Issue",
        "Scheduling Conflict",
        "Personal Issue",
        "Other",
        "Unknown"
    ]

    confidence: float = Field(ge=0.0, le=1.0)

    summary: str = Field(min_length=5, max_length=200)

    recommended_action: str = Field(min_length=5, max_length=200)

print("FollowupInsight schema created successfully.")


FollowupInsight schema created successfully.


In [36]:

def analyze_note(note_text):

    # Handle blank notes
    if pd.isna(note_text) or not str(note_text).strip():
        return {
            "root_cause": "Unknown",
            "confidence": 0.0,
            "summary": "No follow-up note available.",
            "recommended_action": "Contact patient to confirm reason.",
            "validation_status": "skipped_blank",
            "raw_response": None
        }

    prompt = f"""
You are a healthcare appointment follow-up analyst.

Analyze the patient follow-up note below.

Classify the primary reason using ONLY one category:
- No Reminder
- Distance
- Cost
- Health Issue
- Scheduling Conflict
- Personal Issue
- Other
- Unknown

Rules:
1. Do not invent patient information.
2. If the reason is unclear, select Unknown.
3. Confidence must be between 0 and 1.
4. Summary must be concise and factual.
5. Recommended action must be practical.
6. Return only valid JSON.

Required JSON:
{{
    "root_cause": "Category",
    "confidence": 0.0,
    "summary": "Short summary",
    "recommended_action": "Suggested action"
}}

Patient follow-up note:
{str(note_text)}
"""

    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt
        )

        raw_response = response.text

        cleaned_response = raw_response.strip()
        cleaned_response = cleaned_response.replace("```json", "")
        cleaned_response = cleaned_response.replace("```", "").strip()

        parsed = json.loads(cleaned_response)

        insight = FollowupInsight(**parsed)

        return {
            **insight.model_dump(),
            "validation_status": "valid",
            "raw_response": raw_response
        }

    except Exception as e:
        return {
            "root_cause": "Unknown",
            "confidence": 0.0,
            "summary": "AI analysis could not be completed.",
            "recommended_action": "Review note and retry analysis.",
            "validation_status": f"error: {type(e).__name__}",
            "raw_response": None
        }

print("analyze_note() function created successfully.")


analyze_note() function created successfully.


In [37]:

blank_result = analyze_note(np.nan)

print(json.dumps(blank_result, indent=2))


{
  "root_cause": "Unknown",
  "confidence": 0.0,
  "summary": "No follow-up note available.",
  "recommended_action": "Contact patient to confirm reason.",
  "validation_status": "skipped_blank",
  "raw_response": null
}


In [38]:

sample_response = {
    "root_cause": "Distance",
    "confidence": 0.92,
    "summary": "Patient lives far from the clinic.",
    "recommended_action": "Offer a closer clinic or teleconsultation."
}

validated_sample = FollowupInsight(**sample_response)

print(validated_sample.model_dump())


{'root_cause': 'Distance', 'confidence': 0.92, 'summary': 'Patient lives far from the clinic.', 'recommended_action': 'Offer a closer clinic or teleconsultation.'}


## 4.4 Batch Analysis of Follow-up Notes

The analyze_note() function is applied to patient follow-up notes to identify potential reasons for missed appointments.

To manage API usage, only the first 10 notes are analyzed using Gemini.

Blank notes are handled separately, while API failures and invalid responses are recorded without interrupting the entire analysis process.

The generated insights are stored in a structured DataFrame for further validation and export.


In [39]:

import time

# Select the first 10 follow-up notes
sample_notes = notes_df.head(10).copy()

analysis_results = []

for position, (_, row) in enumerate(sample_notes.iterrows(), start=1):

    print(f"Analyzing {position}/10: {row['note_id']}")

    result = analyze_note(row["note_text"])

    result["note_id"] = row["note_id"]
    result["appointment_id"] = row["appointment_id"]

    analysis_results.append(result)

    # Wait between API calls to manage request frequency
    if position < len(sample_notes):
        time.sleep(13)

# Convert results into a DataFrame
notes_analysed_df = pd.DataFrame(analysis_results)

print("\nBatch Analysis Completed.")
print("Records processed:", len(notes_analysed_df))

display(notes_analysed_df)


Analyzing 1/10: N001
Analyzing 2/10: N002
Analyzing 3/10: N003
Analyzing 4/10: N004
Analyzing 5/10: N005
Analyzing 6/10: N006
Analyzing 7/10: N007
Analyzing 8/10: N008
Analyzing 9/10: N009
Analyzing 10/10: N010

Batch Analysis Completed.
Records processed: 10


,root_cause,confidence,summary,recommended_action,validation_status,raw_response,note_id,appointment_id
0,No Reminder,1.00,Patient missed the appointment because she did...,Reschedule the appointment for next week and v...,valid,"{\n ""root_cause"": ""No Reminder"",\n ""conf...",N001,A0872
1,Distance,1.00,Patient missed the appointment due to living f...,Offer telehealth options for routine follow-up...,valid,"{\n ""root_cause"": ""Distance"",\n ""confide...",N002,A0897
2,Cost,1.00,Patient skipped the appointment due to high co...,Provide information on available financial ass...,valid,"{\n ""root_cause"": ""Cost"",\n ""confidence""...",N003,A0526
3,Unknown,0.00,No follow-up note available.,Contact patient to confirm reason.,skipped_blank,None,N004,A0926
4,Health Issue,1.00,Patient missed the appointment due to a high f...,Wait for the patient to recover and contact th...,valid,"{\n ""root_cause"": ""Health Issue"",\n ""con...",N005,A0534
5,Scheduling Conflict,1.00,Patient missed the appointment due to an urgen...,Reschedule the appointment for an evening slot...,valid,"{\n ""root_cause"": ""Scheduling Conflict"",\n ...",N006,A0507
6,No Reminder,0.95,Patient forgot the appointment because they di...,Reschedule the appointment and verify the pati...,valid,"{\n ""root_cause"": ""No Reminder"",\n ""conf...",N007,A0333
7,Distance,0.90,Patient missed the appointment due to a city b...,Reschedule the appointment and offer telehealt...,valid,"{\n ""root_cause"": ""Distance"",\n ""confide...",N008,A0222
8,Unknown,1.00,Patient could not be reached after three attem...,Attempt to contact the patient via an alternat...,valid,"{\n ""root_cause"": ""Unknown"",\n ""confiden...",N009,A0977
9,Scheduling Conflict,0.95,Patient experienced a work shift change preven...,Reschedule the appointment for a Saturday and ...,valid,"{\n ""root_cause"": ""Scheduling Conflict"",\n ...",N010,A0596


In [40]:

# Dataset shape
print("Dataset Shape:", notes_analysed_df.shape)

# Validation status
print("\nValidation Status:")
print(notes_analysed_df["validation_status"].value_counts())

# Root cause distribution
print("\nRoot Cause Distribution:")
print(notes_analysed_df["root_cause"].value_counts())

# Missing values
print("\nMissing Values:")
print(notes_analysed_df.isnull().sum())


Dataset Shape: (10, 8)

Validation Status:
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64

Root Cause Distribution:
root_cause
No Reminder            2
Distance               2
Unknown                2
Scheduling Conflict    2
Cost                   1
Health Issue           1
Name: count, dtype: int64

Missing Values:
root_cause            0
confidence            0
summary               0
recommended_action    0
validation_status     0
raw_response          1
note_id               0
appointment_id        0
dtype: int64


In [41]:

# Export AI analysis results
notes_analysed_df.to_csv(
    "notes_analysed.csv",
    index=False
)

print("notes_analysed.csv saved successfully.")


notes_analysed.csv saved successfully.


In [42]:

from google.colab import files

files.download("notes_analysed.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4.5 Final Insights & Project Conclusion

### Objective
Summarise the key findings from SQL analysis, machine learning prediction, and AI-powered follow-up note analysis.

### Key Insights
- Identify clinics with higher no-show rates.
- Analyse how appointment booking lead time affects attendance.
- Evaluate the performance of machine learning models.
- Identify common reasons for missed appointments using AI-generated insights.
- Recommend actionable strategies to reduce appointment no-shows.

### Business Recommendations
Use the findings to improve reminder strategies, optimise appointment scheduling, and provide personalised patient follow-ups.

In [44]:
print("NoShowGuard – FINAL PROJECT SUMMARY")
print("-" * 45)

# Summarize the follow-up notes and AI analysis since df/X are not in this notebook
print("AI Analysis Dataset:")
print(f"Total follow-up notes loaded: {len(notes_df)}")
if 'notes_analysed_df' in globals():
    print(f"Notes successfully analysed via Gemini: {len(notes_analysed_df)}")
    print("\nAI-Extracted Root Causes (Sample):")
    print(notes_analysed_df["root_cause"].value_counts().to_string())
else:
    print("Notes analyzed: 0 (Batch analysis has not been run yet)")

print("\nAI analysis integration completed successfully.")

NoShowGuard – FINAL PROJECT SUMMARY
---------------------------------------------
AI Analysis Dataset:
Total follow-up notes loaded: 30
Notes successfully analysed via Gemini: 10

AI-Extracted Root Causes (Sample):
root_cause
No Reminder            2
Distance               2
Unknown                2
Scheduling Conflict    2
Cost                   1
Health Issue           1

AI analysis integration completed successfully.


In [45]:
# Save AI analysed notes
notes_df.to_csv("notes_analysed.csv", index=False)

print("File saved: notes_analysed.csv")
print("Shape:", notes_df.shape)

File saved: notes_analysed.csv
Shape: (30, 3)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>